# Projeto 1 – Análise Exploratória ReclameAqui
## Empresa: BigLojas
## Integrantes: Pedro, Victor Rios

## Objetivos de Análise
1. Identificar os principais motivos de reclamação
2. Mapear os defeitos mais frequentes nos produtos
3. Analisar a taxa de resolução por STATUS
4. Identificar os estados com mais reclamações
5. Detectar padrões sazonais (meses com pico de reclamações)

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
import unicodedata
from wordcloud import WordCloud, STOPWORDS

In [ ]:
# Carregamento do dataset
df = pd.read_csv("/content/RECLAMEAQUI_BIGLOJAS.csv")
df.head()

In [ ]:
df.info()

In [ ]:
# Verificar valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())

In [ ]:
# Análise descritiva geral
print("Total de reclamações:", len(df))

print("\nDistribuição por STATUS:")
print(df["STATUS"].value_counts())

print("\nDistribuição por MÊS:")
print(df["MES"].value_counts().sort_index())

## Tratamento de Dados — Novas Colunas

Criamos as seguintes colunas auxiliares para enriquecer a análise:
- `TEMA_CATEGORIA`: categorização automática dos temas por palavras-chave
- `ESTADO`: UF extraída da coluna LOCAL via regex
- `TAMANHO_TEXTO`: comprimento em caracteres da DESCRICAO
- `FAIXA_TAMANHO_TEXTO`: classificação do tamanho em faixas (Curto / Médio / Longo / Muito longo)

In [ ]:
# ── Categorização dos temas ──────────────────────────────────────────────────
def categorizar_tema(texto):
    texto = str(texto).lower()

    if any(p in texto for p in [
        "propaganda", "promoção enganosa", "enganosa", "enganação",
        "promoção falsa", "oferta", "anúncio", "tabloide", "tablóide",
        "preço diferente", "preço incorreto", "preço errado",
        "valor diferente", "valor errado", "preços divergentes",
        "preço cobrado diferente", "desconto", "divergência",
        "preço da prateleira", "preços enganosos", "precificação",
        "sem preço", "produto sem preço", "gôndola",
        "preço promocional não aplicado", "valor mais alto",
        "estoque", "vende produto e depois", "produto anunciado"
    ]):
        return "propaganda/preço enganoso"

    if any(p in texto for p in [
        "carne", "produto vencido", "produtos vencidos", "estragado",
        "podre", "vencido", "embolorado", "mofado", "mofo",
        "sorvete estragado", "palmito", "leite azedo", "ovo podre",
        "queijo", "lasanha", "salgadinho azedo", "azeite", "panetone",
        "alimento", "comida", "peixe", "frango", "linguiça", "salmão",
        "bacalhau", "pernil", "picanha", "inseto", "bicho", "larva",
        "caruncho", "mosca", "verme", "produto estragado",
        "chocolate", "iogurte", "cerveja", "vinho", "uvas podres",
        "reação alérgica", "alérgica", "pão"
    ]):
        return "produto alimentar com problema"

    if any(p in texto for p in [
        "entrega", "pedido", "frete", "transportadora",
        "produto não entregue", "não entregue", "não entregaram",
        "cancelaram meu pedido", "pedido cancelado", "atraso",
        "delivery", "ifood", "não foi entregue",
        "compra cancelada", "não recebi", "nunca chegou"
    ]):
        return "problema na entrega/pedido"

    if any(p in texto for p in [
        "cobrança", "pagamento", "cartão", "reembolso", "nota fiscal",
        "estorno", "duplicidade", "cobrado", "débito", "crédito",
        "segunda via", "2 via", "juros", "parcelamento",
        "valor descontado", "não devolvem", "devolução", "ressarcimento",
        "duplicada", "indevida", "valor ñ", "valor não foi estornado"
    ]):
        return "problema financeiro/cobrança"

    if any(p in texto for p in [
        "atendimento", "suporte", "sac", "mal atendimento",
        "falta de respeito", "descaso", "desrespeito", "absurdo",
        "demora", "enrolação", "funcionário", "funcionaria",
        "gerente", "grosseria", "humilhação", "constrangimento",
        "despreparado", "mal educado", "péssimo atendimento",
        "pessimo atendimento", "fila", "pouco caso", "deboche"
    ]):
        return "mau atendimento"

    if any(p in texto for p in [
        "defeito", "quebrado", "danificado", "garantia", "troca",
        "não funciona", "defeituoso", "qualidade", "geladeira",
        "notebook", "celular", "televisão", "tv ", "máquina",
        "tablet", "impressora", "freezer", "colchão", "cadeira",
        "pneu", "microondas", "ventilador", "refrigerador"
    ]):
        return "produto com defeito/troca"

    if any(p in texto for p in [
        "furto", "furtada", "furtado", "roubo", "assaltado",
        "arrombamento", "segurança", "estacionamento", "bicicleta",
        "ressarcimento furto"
    ]):
        return "segurança/furto"

    if any(p in texto for p in [
        "racismo", "preconceito", "homofobia", "discriminação",
        "machismo", "lgbt", "injúria racial", "bullying"
    ]):
        return "discriminação/preconceito"

    if any(p in texto for p in ["barulho", "poluição sonora", "lei do silêncio"]):
        return "problemas na loja"

    return "outros"


df["TEMA_CATEGORIA"] = df["TEMA"].apply(categorizar_tema)

print("Distribuição por categoria:")
print(df["TEMA_CATEGORIA"].value_counts())

In [ ]:
# ── Coluna ESTADO extraída de LOCAL ─────────────────────────────────────────
df["ESTADO"] = (
    df["LOCAL"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.extract(r'([A-Z]{2})$')
)

print("Estados identificados:")
print(df["ESTADO"].value_counts())

In [ ]:
# ── Tamanho e faixa do texto da descrição ───────────────────────────────────
df["TAMANHO_TEXTO"] = df["DESCRICAO"].fillna("").astype(str).str.len()

def classificar_faixa(valor):
    if valor <= 300:
        return "Curto (0-300)"
    elif valor <= 800:
        return "Médio (301-800)"
    elif valor <= 1500:
        return "Longo (801-1500)"
    else:
        return "Muito longo (1501+)"

df["FAIXA_TAMANHO_TEXTO"] = df["TAMANHO_TEXTO"].apply(classificar_faixa)

print("Estatísticas de tamanho do texto:")
print(df["TAMANHO_TEXTO"].describe().round(1))

print("\nDistribuição por faixa de tamanho:")
print(df["FAIXA_TAMANHO_TEXTO"].value_counts())

## Análise Exploratória (EDA)

In [ ]:
# ── Gráfico 1: Reclamações por mês (sazonalidade) ───────────────────────────
reclamacoes_mes = df.groupby("MES")["CASOS"].sum().sort_index()

plt.figure(figsize=(10, 5))
plt.plot(reclamacoes_mes.index, reclamacoes_mes.values, marker="o")
plt.title("Reclamações por Mês - BigLojas", fontsize=14)
plt.xlabel("Mês")
plt.ylabel("Número de Reclamações")
plt.xticks(reclamacoes_mes.index)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Top 10 motivos de reclamação ──────────────────────────────────
ranking_temas = df["TEMA"].value_counts().head(10)

plt.figure(figsize=(10, 6))
ranking_temas.plot(kind="barh", color="tomato", edgecolor="black")
plt.title("Top 10 Motivos de Reclamação - BigLojas", fontsize=14)
plt.xlabel("Número de Reclamações")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 3: Status × Categoria (barras agrupadas) ────────────────────────
cruzamento = df.groupby(["TEMA_CATEGORIA", "STATUS"]).size().unstack(fill_value=0)

cruzamento.plot(kind="bar", figsize=(12, 6), edgecolor="black")
plt.title("Status das Reclamações por Categoria - BigLojas", fontsize=14)
plt.xlabel("Categoria")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Status")
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 4: Proporção de STATUS (pizza) ───────────────────────────────────
proporcao_status = df["STATUS"].value_counts(normalize=True) * 100

print("Proporção de status (%):")
print(proporcao_status.round(2))

plt.figure(figsize=(8, 8))
df["STATUS"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Proporção de Status das Reclamações - BigLojas", fontsize=14)
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 5: Reclamações por Categoria (barras horizontais) ────────────────
categorias = df["TEMA_CATEGORIA"].value_counts().sort_values()

plt.figure(figsize=(10, 6))
categorias.plot(kind="barh", color="steelblue", edgecolor="black")
plt.title("Reclamações por Categoria - BigLojas", fontsize=14)
plt.xlabel("Quantidade")
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 6: Top 10 estados com mais reclamações (Pareto) ──────────────────
reclamacoes_estado = (
    df.dropna(subset=["ESTADO"])
    .groupby("ESTADO")["CASOS"]
    .sum()
    .sort_values(ascending=False)
)

reclamacoes_estado_pct = reclamacoes_estado.cumsum() / reclamacoes_estado.sum() * 100

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(reclamacoes_estado.index, reclamacoes_estado.values, color="steelblue", label="Casos")
ax1.set_xlabel("Estado")
ax1.set_ylabel("Casos")
ax1.set_title("Distribuição Espacial (Pareto) - BigLojas", fontsize=14)

ax2 = ax1.twinx()
ax2.plot(reclamacoes_estado.index, reclamacoes_estado_pct.values,
         color="darkorange", marker="o", linewidth=2, label="% acumulado")
ax2.set_ylabel("% acumulado")
ax2.set_ylim(0, 110)

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
plt.tight_layout()
plt.show()

print("\nTop 5 estados:")
print(reclamacoes_estado.head())

In [ ]:
# ── Gráfico 7: Histograma — Tamanho da Descrição × Status ────────────────────
status_lista = df["STATUS"].unique()
cores = ["steelblue", "tomato", "seagreen", "darkorange", "purple"]

plt.figure(figsize=(12, 5))
for i, status in enumerate(status_lista):
    subset = df[df["STATUS"] == status]["TAMANHO_TEXTO"]
    plt.hist(subset, bins=30, alpha=0.5, label=status, color=cores[i % len(cores)])

plt.title("Histograma — Tamanho da Descrição × Status", fontsize=14)
plt.xlabel("Tamanho da descrição (caracteres)")
plt.ylabel("Frequência")
plt.legend(title="STATUS")
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 8: Boxplot — Tamanho da Descrição × Status ───────────────────────
status_order = df["STATUS"].value_counts().index.tolist()
grupos = [df[df["STATUS"] == s]["TAMANHO_TEXTO"].dropna().values for s in status_order]

plt.figure(figsize=(12, 6))
plt.boxplot(grupos, labels=status_order, patch_artist=True)
plt.title("Boxplot — Tamanho da Descrição × Status", fontsize=14)
plt.xlabel("Status")
plt.ylabel("Tamanho da descrição (caracteres)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 9: Faixas de Tamanho do Texto ────────────────────────────────────
ordem_faixas = ["Curto (0-300)", "Médio (301-800)", "Longo (801-1500)", "Muito longo (1501+)"]
faixas_counts = df["FAIXA_TAMANHO_TEXTO"].value_counts().reindex(ordem_faixas, fill_value=0)

plt.figure(figsize=(9, 5))
bars = plt.bar(faixas_counts.index, faixas_counts.values, color="steelblue", edgecolor="black")
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontsize=11
    )
plt.title("Faixas de Tamanho do Texto", fontsize=14)
plt.xlabel("Faixa de tamanho")
plt.ylabel("Quantidade de reclamações")
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 10: WordCloud das Descrições ─────────────────────────────────────
stopwords_pt = {
    "a", "o", "e", "de", "do", "da", "dos", "das", "um", "uma", "uns", "umas",
    "em", "no", "na", "nos", "nas", "para", "por", "com", "sem", "que", "eu",
    "me", "minha", "meu", "meus", "minhas", "se", "ao", "aos", "à", "às", "foi",
    "são", "ser", "ter", "tem", "mais", "muito", "pouco", "já", "não", "sim",
    "como", "quando", "onde", "porque", "pois", "ou", "mas", "também", "sobre",
    "após", "antes", "entre", "até", "há", "isso", "isto", "aquele", "aquela",
    "eles", "elas", "ela", "ele", "cliente", "empresa", "loja", "biglojas",
    "reclameaqui", "produto", "atendimento", "fui", "vou", "está", "estao",
    "estava", "pra", "pro", "q", "pq", "dia", "dias", "nao", "big"
}
todas_stopwords = set(STOPWORDS).union(stopwords_pt)

texto_unico = " ".join(df["DESCRICAO"].dropna().astype(str).tolist())
texto_unico = texto_unico.lower()
texto_unico = unicodedata.normalize("NFKD", texto_unico).encode("ascii", "ignore").decode("utf-8")
texto_unico = re.sub(r"http\S+", " ", texto_unico)
texto_unico = re.sub(r"[^a-zA-Z\s]", " ", texto_unico)
texto_unico = re.sub(r"\s+", " ", texto_unico).strip()

wc = WordCloud(
    width=1200, height=600, background_color="white",
    stopwords=todas_stopwords, collocations=False, max_words=120
).generate(texto_unico)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud das Descrições - BigLojas", fontsize=14)
plt.tight_layout()
plt.show()

## Resumo Estatístico Final

In [ ]:
print("="*50)
print("RESUMO ESTATÍSTICO — BigLojas")
print("="*50)
print(f"Total de reclamações : {len(df)}")
print(f"Total de casos       : {int(df['CASOS'].sum())}")
print(f"Status mais comum    : {df['STATUS'].value_counts().idxmax()}")
print(f"Estado líder         : {df['ESTADO'].value_counts().idxmax()}")
print(f"Categoria principal  : {df['TEMA_CATEGORIA'].value_counts().index[1]}")
print(f"Média tamanho texto  : {int(df['TAMANHO_TEXTO'].mean())} caracteres")
print(f"Mês com mais casos   : {df.groupby('MES')['CASOS'].sum().idxmax()}")